# Anomaly-Transformer TL — Temperatura SIATA Estación 203

Transfer Learning del modelo **Anomaly-Transformer** para detección de anomalías de temperatura en la estación UNAN (código 203) del sistema SIATA.

**Período:** 2023-01-01 → 2023-06-30  
**Estrategia:** Freeze schedule (encoder congelado primeras 20 épocas, fine-tuning completo épocas 21-100)  
**Métrica principal:** Balanced Accuracy

In [ ]:
# Celda 1: Configuración GitHub / Colab
import os, sys

REPO_URL = 'https://github.com/ronvas234/data-science-monograph.git'
REPO_DIR = 'data-science-monograph'
BRANCH   = 'feature/add-transforme'

# Instalar git-lfs para poder descargar el CSV grande
os.system('apt-get install -y git-lfs 2>/dev/null')

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {BRANCH} {REPO_URL}')

if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

# Descargar objetos LFS (CSV de temperatura)
os.system('git lfs pull')

AT_PATH = os.path.abspath('modelos/Anomaly-Transformer')
if AT_PATH not in sys.path:
    sys.path.insert(0, AT_PATH)

print(f'CWD: {os.getcwd()}')
print(f'AT path en sys.path: {AT_PATH}')

In [ ]:
# Celda 2: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_curve, auc, balanced_accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Celda 3: Configuración de hiperparámetros
STATION_CODE = 203
FECHA_INICIO = '2023-01-01'
FECHA_FIN    = '2023-06-30'

WIN_SIZE     = 100   # ventana de 100 minutos (~1.67 h)
BATCH_SIZE   = 256
N_EPOCHS_P1  = 20    # Fase 1: encoder congelado
N_EPOCHS_P2  = 80    # Fase 2: fine-tuning completo
N_EPOCHS     = N_EPOCHS_P1 + N_EPOCHS_P2
K            = 3     # peso de la pérdida KL
LR_P1        = 1e-3  # lr fase 1 (capas externas)
LR_P2        = 1e-4  # lr fase 2 (fine-tuning)
PATIENCE     = 10    # early stopping (solo en fase 2)

# Arquitectura del modelo (adaptada para 1 canal)
D_MODEL  = 64
N_HEADS  = 4
E_LAYERS = 3
D_FF     = 64
DROPOUT  = 0.1

# Split cronológico
TRAIN_RATIO = 0.65
VAL_RATIO   = 0.20
# TEST_RATIO = 0.15

CSV_PATH = 'modelos/Anomaly-Transformer/data/temperatura_estaciones_2020_2025.csv'

print(f'Estación: {STATION_CODE} (UNAN) | Período: {FECHA_INICIO} -> {FECHA_FIN}')
print(f'Ventana: {WIN_SIZE} min | Épocas: P1={N_EPOCHS_P1} + P2={N_EPOCHS_P2}')
print(f'Modelo: d_model={D_MODEL}, n_heads={N_HEADS}, e_layers={E_LAYERS}')

In [ ]:
# Celda 4: VisualizationService

class VisualizationService:

    @staticmethod
    def plot_time_series(df, df_train, df_val, df_test, station_code, fecha_inicio, fecha_fin):
        """Grafica 1: Serie temporal con split Train/Val/Test y puntos de anomalia."""
        fig, ax = plt.subplots(figsize=(16, 4))
        ax.plot(df_train['fecha_hora'], df_train['t'], color='steelblue', lw=0.6, label='Train')
        ax.plot(df_val['fecha_hora'],   df_val['t'],   color='orange',    lw=0.6, label='Val')
        ax.plot(df_test['fecha_hora'],  df_test['t'],  color='green',     lw=0.6, label='Test')
        anom = df[df['anomaly'] == 1]
        ax.scatter(anom['fecha_hora'], anom['t'], color='purple', s=8, zorder=5, label='Anomalia')
        ax.set_title(f'Serie temporal — Sensor SIATA {station_code} ({fecha_inicio} -> {fecha_fin})')
        ax.set_xlabel('Fecha')
        ax.set_ylabel('Temperatura (°C)')
        ax.legend(loc='upper right')
        plt.tight_layout()
        plt.savefig('grafica_1_serie_temporal.png', dpi=150, bbox_inches='tight')
        plt.show()

    @staticmethod
    def plot_training_curve(history, freeze_epoch):
        """Grafica 2: Curvas val_loss1/val_loss2 con linea de descongelado."""
        fig, ax = plt.subplots(figsize=(12, 5))
        epochs = range(1, len(history['val_loss1']) + 1)
        ax.plot(epochs, history['val_loss1'], label='val_loss1 (series)', color='steelblue')
        ax.plot(epochs, history['val_loss2'], label='val_loss2 (prior)',  color='orange')
        ax.axvline(x=freeze_epoch, linestyle='--', color='gray',
                   label=f'Descongelado (época {freeze_epoch})')
        ax.set_xlabel('Época')
        ax.set_ylabel('Pérdida (MSE + KL)')
        ax.set_title('Curva de entrenamiento — TL + Freeze Schedule')
        ax.legend()
        plt.tight_layout()
        plt.savefig('grafica_2_training_curve.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Mejor val_loss combinado registrado durante entrenamiento')

    @staticmethod
    def plot_roc_threshold(energy, labels):
        """Grafica 3: Curva ROC con umbral optimo (Youden J). Retorna umbral."""
        fpr, tpr, thresholds = roc_curve(labels, energy)
        roc_auc = auc(fpr, tpr)
        J = tpr - fpr
        opt_idx = int(np.argmax(J))
        umbral  = float(thresholds[opt_idx])
        print(f'Umbral optimo (Youden J): {umbral:.6f} | '
              f'TPR={tpr[opt_idx]:.4f} FPR={fpr[opt_idx]:.4f} AUC={roc_auc:.4f}')
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.plot(fpr, tpr, label=f'ROC Curve (AUC={roc_auc:.4f})', color='steelblue')
        ax.scatter([fpr[opt_idx]], [tpr[opt_idx]], color='red', zorder=5,
                   label=f"Youden J (thr={umbral:.6f})")
        ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Aleatorio')
        ax.set_xlabel('FPR (1 - Especificidad)')
        ax.set_ylabel('TPR (Sensibilidad)')
        ax.set_title(f'ROC Curve — Anomaly-Transformer TL Sensor {STATION_CODE}')
        ax.legend()
        plt.tight_layout()
        plt.savefig('grafica_3_roc.png', dpi=150, bbox_inches='tight')
        plt.show()
        return umbral

    @staticmethod
    def plot_error_reconstruction(timestamps, energy, labels, umbral, label, show_pred=False):
        """Graficas 4 y 5: Error de reconstruccion con umbral y anomalias reales/predichas."""
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(timestamps, energy, color='steelblue', lw=0.5, label='Error de reconstruccion')
        anom_mask = labels == 1
        ax.scatter(timestamps[anom_mask], energy[anom_mask],
                   color='red', s=14, zorder=5, label='Anomalias reales')
        if show_pred:
            pred_mask = energy > umbral
            ax.scatter(timestamps[pred_mask], energy[pred_mask],
                       color='purple', marker='x', s=20, zorder=6, label='Predichas')
        ax.axhline(y=umbral, linestyle='--', color='black', lw=1.2,
                   label=f'Umbral theta = {umbral:.4f}')
        ax.set_xlabel('Tiempo')
        ax.set_ylabel('Error')
        ax.set_title(f'Errores de reconstruccion — conjunto {label}')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%Y'))
        ax.legend(fontsize=8)
        plt.tight_layout()
        fname = f'grafica_error_recon_{label.lower()}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()

    @staticmethod
    def plot_reconstruction(timestamps, original, reconstructed, labels, pred=None, label=''):
        """Graficas 6, 7, 8: Serie reconstruida vs original."""
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(timestamps, original,      color='steelblue', lw=0.7, label='Original')
        ax.plot(timestamps, reconstructed, color='orange',    lw=0.7, label='Reconstruida', alpha=0.85)
        anom_mask = labels[:len(timestamps)] == 1
        ax.scatter(timestamps[anom_mask], original[anom_mask],
                   color='red', s=12, zorder=5, label='Anomalias reales')
        if pred is not None:
            pred_mask = pred[:len(timestamps)] == 1
            ax.scatter(timestamps[pred_mask], reconstructed[pred_mask],
                       color='purple', marker='x', s=20, zorder=6, label='Anomalias predichas')
        ax.set_xlabel('Fecha')
        ax.set_ylabel('Temperatura (°C)')
        ax.set_title(f'Reconstruccion vs Original + Anomalias — AT TL {label}')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%Y'))
        ax.legend(fontsize=8)
        plt.tight_layout()
        fname = f'grafica_recon_{label.lower().replace(" ", "_")}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()

    @staticmethod
    def print_metrics(metrics):
        """Grafica 9: Imprime metricas de clasificacion y matriz de confusion."""
        print(f'Accuracy:     {metrics["accuracy"]:.4f}')
        print(f'Balanced Acc: {metrics["balanced_acc"]:.4f}')
        print(f'Precision: {metrics["precision"]:.4f}')
        print(f'Recall:    {metrics["recall"]:.4f}')
        print(f'F1:        {metrics["f1"]:.4f}')
        print()
        cm = metrics['confusion_matrix']
        cm_df = pd.DataFrame(
            cm,
            index=['Pred Normal', 'Pred Anomalia'],
            columns=['Real Normal', 'Real Anomalia']
        )
        print('Confusion Matrix:')
        print(cm_df.to_string())
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Normal', 'Anomalia'],
                    yticklabels=['Normal', 'Anomalia'])
        ax.set_xlabel('Real')
        ax.set_ylabel('Predicho')
        ax.set_title(f'Confusion Matrix — AT TL Sensor {STATION_CODE}')
        plt.tight_layout()
        plt.savefig('grafica_9_confusion_matrix.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# Celda 5: EvaluationService

class EvaluationService:

    @staticmethod
    def compute_energy(model, loader, device):
        """MSE de reconstruccion por ventana + labels. Sin bias por batch-softmax."""
        model.eval()
        energies, labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in loader:
                X_batch = X_batch.to(device)
                output, _, _, _ = model(X_batch)
                rec_err = torch.mean((output - X_batch) ** 2, dim=(1, 2))
                energies.extend(rec_err.cpu().numpy().tolist())
                labels.extend(y_batch.numpy().tolist())
        return np.array(energies, dtype=np.float32), np.array(labels, dtype=np.int32)

    @staticmethod
    def compute_metrics(energy, labels, umbral):
        """Metricas de clasificacion binaria. Balanced Accuracy como metrica principal."""
        pred = (energy > umbral).astype(int)
        cm   = confusion_matrix(labels, pred, labels=[0, 1])
        return {
            'accuracy':     float(np.mean(pred == labels)),
            'balanced_acc': balanced_accuracy_score(labels, pred),
            'precision':    precision_score(labels, pred, zero_division=0),
            'recall':       recall_score(labels, pred, zero_division=0),
            'f1':           f1_score(labels, pred, zero_division=0),
            'confusion_matrix': cm
        }

    @staticmethod
    def compute_reconstruction(model, df_segment, scaler, batch_size, device, win_size):
        """
        Para cada ventana j, reconstruye el primer timestep (indice 0).
        Retorna: (timestamps, original_°C, reconstruido_°C) de longitud N-win_size+1.
        La normalizacion se aplica solo con el scaler ajustado en Train.
        """
        vals = df_segment[['t']].values.astype(np.float32)
        n    = len(vals)
        if n < win_size:
            return np.array([]), np.array([]), np.array([])

        scaled = scaler.transform(vals)
        X = np.array([scaled[i:i+win_size] for i in range(n - win_size + 1)], dtype=np.float32)
        loader_r = DataLoader(TensorDataset(torch.tensor(X)), batch_size=batch_size, shuffle=False)

        model.eval()
        recons = []
        with torch.no_grad():
            for (Xb,) in loader_r:
                Xb  = Xb.to(device)
                out, _, _, _ = model(Xb)
                # Invertir normalizacion: primer timestep de cada ventana
                r = scaler.inverse_transform(out[:, 0, :].cpu().numpy())[:, 0]
                recons.extend(r.tolist())

        n_win = len(recons)
        timestamps   = df_segment['fecha_hora'].values[:n_win]
        original_deg = df_segment['t'].values[:n_win]
        return timestamps, original_deg, np.array(recons, dtype=np.float32)

In [ ]:
# Celda 6: Carga y exploracion de datos

print('Cargando CSV...')
df_raw = pd.read_csv(CSV_PATH, parse_dates=['fecha_hora'])

# Filtrar estacion y rango temporal
df = df_raw[
    (df_raw['codigo'] == STATION_CODE) &
    (df_raw['fecha_hora'] >= FECHA_INICIO) &
    (df_raw['fecha_hora'] <= FECHA_FIN)
].copy()

df = df.sort_values('fecha_hora').reset_index(drop=True)
df = df.dropna(subset=['t'])

# Asegurar tipos correctos para columnas booleanas
for col in ['calidad_dudosa', 'temperatura_dudosa']:
    if df[col].dtype == object:
        df[col] = df[col].map({'True': True, 'False': False, True: True, False: False}).fillna(False)
    df[col] = df[col].astype(bool)

# Label de anomalia: calidad_dudosa OR temperatura_dudosa
df['anomaly'] = (df['calidad_dudosa'] | df['temperatura_dudosa']).astype(int)

print(f'Registros cargados: {len(df):,}')
print(f'Periodo: {df["fecha_hora"].min()} -> {df["fecha_hora"].max()}')
print(f'Temperatura: min={df["t"].min():.1f} C | max={df["t"].max():.1f} C | mean={df["t"].mean():.1f} C')
print(f'Anomalias (puntos): {df["anomaly"].sum():,} ({100*df["anomaly"].mean():.2f}%)')
df[['fecha_hora', 't', 'calidad_dudosa', 'temperatura_dudosa', 'anomaly']].head(8)

In [ ]:
# Celda 7: Split cronologico (sin shuffle — evitar sesgo)

n         = len(df)
train_end = int(n * TRAIN_RATIO)
val_end   = int(n * (TRAIN_RATIO + VAL_RATIO))

df_train = df.iloc[:train_end].copy().reset_index(drop=True)
df_val   = df.iloc[train_end:val_end].copy().reset_index(drop=True)
df_test  = df.iloc[val_end:].copy().reset_index(drop=True)

print('=== Split cronologico ===')
for nombre, d in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f'{nombre}: {len(d):,} pts | '
          f'{d["fecha_hora"].iloc[0].date()} -> {d["fecha_hora"].iloc[-1].date()} | '
          f'Anomalias: {d["anomaly"].sum():,} ({100*d["anomaly"].mean():.2f}%)')

In [ ]:
# Celda 8: Grafica 1 — Serie temporal Train/Val/Test + Anomalias

VisualizationService.plot_time_series(
    df, df_train, df_val, df_test,
    STATION_CODE, FECHA_INICIO, FECHA_FIN
)

In [ ]:
# Celda 9: Preprocesamiento — Z-score ajustado SOLO en Train
# (StandardScaler es la normalizacion nativa del Anomaly-Transformer)

scaler = StandardScaler()
scaler.fit(df_train[['t']].values)  # Fit solo en train: evita data leakage

train_scaled = scaler.transform(df_train[['t']].values).astype(np.float32)
val_scaled   = scaler.transform(df_val[['t']].values).astype(np.float32)
test_scaled  = scaler.transform(df_test[['t']].values).astype(np.float32)

print(f'Normalizacion: media={scaler.mean_[0]:.4f} C | std={np.sqrt(scaler.var_[0]):.4f} C')
print(f'Train escalado: [{train_scaled.min():.3f}, {train_scaled.max():.3f}]')
print(f'Val   escalado: [{val_scaled.min():.3f}, {val_scaled.max():.3f}]')
print(f'Test  escalado: [{test_scaled.min():.3f}, {test_scaled.max():.3f}]')

In [ ]:
# Celda 10: Ventanas deslizantes y DataLoaders

def create_windows(scaled_data, labels, win_size):
    """stride=1. Label de ventana = 1 si cualquier punto en la ventana es anomalia."""
    n = len(scaled_data)
    X, y = [], []
    for i in range(n - win_size + 1):
        X.append(scaled_data[i:i+win_size])
        y.append(int(labels[i:i+win_size].max()))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

train_labels = df_train['anomaly'].values
val_labels   = df_val['anomaly'].values
test_labels  = df_test['anomaly'].values

print('Creando ventanas...')
X_train, y_train = create_windows(train_scaled, train_labels, WIN_SIZE)
X_val,   y_val   = create_windows(val_scaled,   val_labels,   WIN_SIZE)
X_test,  y_test  = create_windows(test_scaled,  test_labels,  WIN_SIZE)

print(f'Ventanas Train: {len(X_train):,} | Anomalias: {y_train.sum():,} ({100*y_train.mean():.2f}%)')
print(f'Ventanas Val:   {len(X_val):,}   | Anomalias: {y_val.sum():,}   ({100*y_val.mean():.2f}%)')
print(f'Ventanas Test:  {len(X_test):,}  | Anomalias: {y_test.sum():,}  ({100*y_test.mean():.2f}%)')

def make_loader(X, y, batch_size, shuffle=False):
    return DataLoader(
        TensorDataset(torch.tensor(X), torch.tensor(y)),
        batch_size=batch_size, shuffle=shuffle
    )

train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=False)
val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE)
test_loader  = make_loader(X_test,  y_test,  BATCH_SIZE)
print(f'DataLoaders listos | batch_size={BATCH_SIZE}')

In [ ]:
# Celda 11: Modelo Anomaly-Transformer y funciones de freeze

from model.AnomalyTransformer import AnomalyTransformer

model = AnomalyTransformer(
    win_size=WIN_SIZE,
    enc_in=1,        # 1 canal: temperatura
    c_out=1,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    e_layers=E_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    activation='gelu'
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parametros totales: {total_params:,}')
print(f'Componentes: embedding={sum(p.numel() for p in model.embedding.parameters()):,} | '
      f'encoder={sum(p.numel() for p in model.encoder.parameters()):,} | '
      f'projection={sum(p.numel() for p in model.projection.parameters()):,}')


def freeze_encoder(m):
    """Fase 1: congela encoder, entrena embedding + projection."""
    for p in m.parameters():
        p.requires_grad = False
    for p in m.embedding.parameters():
        p.requires_grad = True
    for p in m.projection.parameters():
        p.requires_grad = True


def unfreeze_all(m):
    """Fase 2: descongela todos los parametros."""
    for p in m.parameters():
        p.requires_grad = True


freeze_encoder(model)
trainable_p1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nFase 1 — parametros entrenables: {trainable_p1:,} (encoder congelado)')

In [ ]:
# Celda 12: Funciones de perdida (minimax KL + MSE)

def kl_loss(p, q):
    """KL(p || q). p y q: pesos de atencion (B, H, L, L). Retorna (B,)."""
    return torch.mean(
        torch.sum(p * (torch.log(p + 1e-4) - torch.log(q + 1e-4)), dim=-1),
        dim=(1, 2)
    )


def normalize_prior(prior):
    """Normaliza el prior gaussiano para que sume 1 en la ultima dimension."""
    s = prior.sum(dim=-1, keepdim=True)
    return prior / (s + 1e-8)


def compute_loss_batch(m, X_batch, k=K):
    """Calcula (loss1, loss2, rec_loss) para un batch. loss1 y loss2 son la perdida minimax."""
    X = X_batch.to(DEVICE)
    output, series, prior, _ = m(X)

    rec_loss = nn.MSELoss()(output, X)

    series_kl = torch.zeros(X.shape[0], device=DEVICE)
    prior_kl  = torch.zeros(X.shape[0], device=DEVICE)

    for u in range(len(prior)):
        p_norm = normalize_prior(prior[u])
        series_kl += kl_loss(series[u], p_norm)
        prior_kl  += kl_loss(p_norm, series[u])

    series_kl = series_kl.mean() / len(prior)
    prior_kl  = prior_kl.mean()  / len(prior)

    # loss1: series quiere seguir la gaussiana (comportamiento normal)
    # loss2: prior quiere alejarse de la serie (sensibilidad a anomalias)
    loss1 = rec_loss - k * series_kl
    loss2 = rec_loss + k * prior_kl

    return loss1, loss2, rec_loss.item()


print('Funciones de perdida definidas.')
print('loss1 = rec_loss - k * KL(series || prior_gaussiana)')
print('loss2 = rec_loss + k * KL(prior_gaussiana || series)')

In [ ]:
# Celda 13: Loop de entrenamiento con freeze schedule y early stopping

history = {'loss1': [], 'loss2': [], 'val_loss1': [], 'val_loss2': []}
best_val_combined = float('inf')
best_state        = None
patience_counter  = 0
stopped_epoch     = N_EPOCHS

# Optimizer Fase 1 (solo capas descongeladas)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_P1
)

for epoch in range(1, N_EPOCHS + 1):

    # ── Transicion a Fase 2 ──
    if epoch == N_EPOCHS_P1 + 1:
        unfreeze_all(model)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR_P2)
        n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'\n[Epoca {epoch}] Encoder descongelado — {n_train:,} params entrenables (lr={LR_P2})')

    # ── Entrenamiento ──
    model.train()
    ep_l1, ep_l2 = [], []
    for X_batch, _ in train_loader:
        loss1, loss2, _ = compute_loss_batch(model, X_batch)
        optimizer.zero_grad()
        loss1.backward(retain_graph=True)
        loss2.backward()
        optimizer.step()
        ep_l1.append(loss1.item())
        ep_l2.append(loss2.item())

    # ── Validacion ──
    model.eval()
    vl1, vl2 = [], []
    with torch.no_grad():
        for X_batch, _ in val_loader:
            l1, l2, _ = compute_loss_batch(model, X_batch)
            vl1.append(l1.item())
            vl2.append(l2.item())

    mean_vl1 = float(np.mean(vl1))
    mean_vl2 = float(np.mean(vl2))
    val_combined = mean_vl1 + mean_vl2

    history['loss1'].append(float(np.mean(ep_l1)))
    history['loss2'].append(float(np.mean(ep_l2)))
    history['val_loss1'].append(mean_vl1)
    history['val_loss2'].append(mean_vl2)

    # ── Checkpoint del mejor modelo ──
    if val_combined < best_val_combined:
        best_val_combined = val_combined
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        if epoch > N_EPOCHS_P1:   # early stopping solo en fase 2
            patience_counter += 1

    if epoch % 10 == 0 or epoch == N_EPOCHS_P1:
        print(f'Epoca {epoch:3d} | vl1={mean_vl1:.4f} | vl2={mean_vl2:.4f} | patience={patience_counter}')

    if patience_counter >= PATIENCE and epoch > N_EPOCHS_P1:
        print(f'\n[Early stopping en epoca {epoch}]')
        stopped_epoch = epoch
        break

print(f'\nMejor val_loss combinado: {best_val_combined:.6f}')
# Restaurar el mejor modelo
model.load_state_dict(best_state)
model.to(DEVICE)
print('Mejor modelo restaurado.')

In [ ]:
# Celda 14: Grafica 2 — Curva de entrenamiento + Freeze Schedule

VisualizationService.plot_training_curve(history, freeze_epoch=N_EPOCHS_P1)

In [ ]:
# Celda 15: Compute energy (MSE de reconstruccion por ventana)
# Metrica principal de anomalia: sin bias por batch-softmax

print('Calculando scores de anomalia...')
energy_train, labels_train = EvaluationService.compute_energy(model, train_loader, DEVICE)
energy_val,   labels_val   = EvaluationService.compute_energy(model, val_loader,   DEVICE)
energy_test,  labels_test  = EvaluationService.compute_energy(model, test_loader,  DEVICE)

print(f'Energy Train | min={energy_train.min():.4f} max={energy_train.max():.4f} mean={energy_train.mean():.4f}')
print(f'Energy Val   | min={energy_val.min():.4f}   max={energy_val.max():.4f}   mean={energy_val.mean():.4f}')
print(f'Energy Test  | min={energy_test.min():.4f}  max={energy_test.max():.4f}  mean={energy_test.mean():.4f}')

In [ ]:
# Celda 16: Umbral optimo — Youden J sobre Val (NO sobre Test)
# El test nunca se toca para seleccion de umbral: evita sesgo de evaluacion

umbral = VisualizationService.plot_roc_threshold(energy_val, labels_val)
print(f'Umbral seleccionado: {umbral:.6f}')

In [ ]:
# Celda 17: Grafica 4 — Errores de reconstruccion — Validacion (sin predicciones)

# Timestamps: ultimo timestep de cada ventana (posicion WIN_SIZE-1 en adelante)
val_ts = df_val['fecha_hora'].values[WIN_SIZE - 1:WIN_SIZE - 1 + len(energy_val)]

VisualizationService.plot_error_reconstruction(
    timestamps=val_ts,
    energy=energy_val,
    labels=labels_val,
    umbral=umbral,
    label='Validacion',
    show_pred=False
)

In [ ]:
# Celda 18: Grafica 5 — Errores de reconstruccion — Test (con anomalias reales)

test_ts = df_test['fecha_hora'].values[WIN_SIZE - 1:WIN_SIZE - 1 + len(energy_test)]

VisualizationService.plot_error_reconstruction(
    timestamps=test_ts,
    energy=energy_test,
    labels=labels_test,
    umbral=umbral,
    label='Test',
    show_pred=True
)

In [ ]:
# Celda 19: Compute reconstruction (serie reconstruida en escala original °C)

print('Calculando series reconstruidas...')
ts_train, orig_train, recon_train = EvaluationService.compute_reconstruction(
    model, df_train, scaler, BATCH_SIZE, DEVICE, WIN_SIZE
)
ts_val, orig_val, recon_val = EvaluationService.compute_reconstruction(
    model, df_val, scaler, BATCH_SIZE, DEVICE, WIN_SIZE
)
ts_test, orig_test, recon_test = EvaluationService.compute_reconstruction(
    model, df_test, scaler, BATCH_SIZE, DEVICE, WIN_SIZE
)

print(f'Puntos reconstruidos: Train={len(ts_train):,} | Val={len(ts_val):,} | Test={len(ts_test):,}')

In [ ]:
# Celda 20: Grafica 6 — Reconstruccion vs Original (Train)

train_labels_pts = df_train['anomaly'].values[:len(ts_train)]

VisualizationService.plot_reconstruction(
    timestamps=ts_train,
    original=orig_train,
    reconstructed=recon_train,
    labels=train_labels_pts,
    pred=None,
    label='Train'
)

In [ ]:
# Celda 21: Grafica 7 — Reconstruccion vs Original (Validacion, sin predicciones)

val_labels_pts = df_val['anomaly'].values[:len(ts_val)]

VisualizationService.plot_reconstruction(
    timestamps=ts_val,
    original=orig_val,
    reconstructed=recon_val,
    labels=val_labels_pts,
    pred=None,
    label='Validacion'
)

In [ ]:
# Celda 22: Grafica 8 — Reconstruccion vs Original (Test con predicciones)

test_labels_pts = df_test['anomaly'].values[:len(ts_test)]

# Prediccion por ventana -> prediccion por punto (primer timestep de cada ventana)
pred_windows = (energy_test > umbral).astype(int)
pred_pts     = np.zeros(len(ts_test), dtype=int)
n_win_pred   = min(len(pred_windows), len(pred_pts))
pred_pts[:n_win_pred] = pred_windows[:n_win_pred]

VisualizationService.plot_reconstruction(
    timestamps=ts_test,
    original=orig_test,
    reconstructed=recon_test,
    labels=test_labels_pts,
    pred=pred_pts,
    label='Test'
)

In [ ]:
# Celda 23: Grafica 9 — Metricas de clasificacion (Balanced Accuracy como principal)

print('=== Metricas — Conjunto de Validacion ===')
metrics_val = EvaluationService.compute_metrics(energy_val, labels_val, umbral)
VisualizationService.print_metrics(metrics_val)

print('\n=== Metricas — Conjunto de Test ===')
metrics_test = EvaluationService.compute_metrics(energy_test, labels_test, umbral)
VisualizationService.print_metrics(metrics_test)